# 原生几何（Primitive-Native）DSL 演示

用手写 primitive 定义 Q1 + Q2 + bus 的几何，导出到 Metal QDesign。

对应 YAML：`chain_2q_native.metal.yaml`

如果你第一次接触 DSL v3，建议先打开 `native_2q_minimal.metal.yaml` 读一遍——
它更短，适合快速理解 vars / hamiltonian / circuit / netlist / geometry 五个顶层 section。

In [1]:
from pathlib import Path
import sys

_SRC = Path("../../src").resolve()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from qiskit_metal.toolbox_metal.design_dsl import build_ir, build_design

## 1. 解析 YAML → DesignIR

`build_ir()` 读取 YAML，做变量插值（`${vars.pad_w}` → 实际值）、
`$extend` 模板展开、primitive/pin 结构化，返回一个 `DesignIR` 对象。
这一步不创建 Metal QDesign，纯粹是 YAML → IR。

In [2]:
ir = build_ir("chain_2q_native.metal.yaml")

print(f"schema:     {ir.schema}")
print(f"components: {[c.name for c in ir.components]}")

schema:     qiskit-metal/design-dsl/3
components: ['Q1', 'Q2', 'bus']


## 2. IR → Metal QDesign

`build_design()` 在 IR 基础上创建标准 Metal `QDesign`：
- primitive → `design.qgeometry.tables`
- pin → `design.components[name].pins`
- 连接 → `design.net_info`

In [3]:
design = build_design("chain_2q_native.metal.yaml")

for table in ("poly", "path", "junction"):
    n = len(design.qgeometry.tables[table])
    print(f"{table:10s}: {n} 行")



poly      : 6 行
path      : 2 行
junction  : 2 行


In [4]:
from qiskit_metal import MetalGUI
gui = MetalGUI(design)

看一下 poly 表的内容——每一行对应一个 primitive：

In [7]:
design.qgeometry.tables["poly"][["component", "name", "subtract", "layer"]]

,component,name,subtract,layer
0,1,pad_left,False,1
1,1,pad_right,False,1
2,1,pocket,True,1
3,2,pad_left,False,1
4,2,pad_right,False,1
5,2,pocket,True,1


## 3. 连接关系

YAML 里定义了两条连接（Q1.bus ↔ bus.start，bus.end ↔ Q2.bus），
导出后出现在 `design.net_info`——每条连接两端各一行，共四行。

In [9]:
design.net_info

,net_id,component_id,pin_name
0,1,1,bus
1,1,3,start
2,2,3,end
3,2,2,bus


## 4. Derived metadata

`design.metadata["dsl_chain"]` 保存了 DSL 完整的解析链路。
`derived` 里包含自动计算的值，比如 bus path 的长度。

In [5]:
chain = design.metadata["dsl_chain"]
print(f"metadata 顶层 keys: {sorted(chain)}")

bus_len = chain["derived"]["circuit"]["geometry"]["bus"]["primitives"]["center_trace"]["length"]
print(f"bus center_trace 长度: {bus_len:.4f} mm")

metadata 顶层 keys: ['circuit', 'derived', 'design', 'geometry', 'hamiltonian', 'netlist', 'schema', 'vars']
bus center_trace 长度: 1.7200 mm


## 5. 检查

YAML 或导出逻辑有变动时，这些断言会提醒你。

In [6]:
assert len(ir.components) == 3, "应该有 Q1, Q2, bus 三个 component"
assert len(design.qgeometry.tables["poly"]) >= 4
assert len(design.qgeometry.tables["path"]) >= 1
assert len(design.net_info) == 4
assert bus_len > 0

print("全部通过")

全部通过
